# CEG-WM Colab 完整工作流执行指南

本 Notebook 在 Google Colab 上执行完整的机器学习工作流（embed/detect/calibrate/evaluate），生成新的 run_root 目录并下载最终结果。

**执行流水线**：
- **Embed 阶段**：对输入进行特征提取和防水印嵌入
- **Detect 阶段**：检测防水印信号
- **Calibrate 阶段**：校准检测阈值和概率
- **Evaluate 阶段**：评估整个工作流的性能

## 第 1 步：上传和解压代码包

将 CEG-WM 代码上传到 Colab。

**方式：从本地上传**
- 直接在 Colab UI 中上传 CEG-WM.zip

In [ ]:
import sys
import os
from pathlib import Path
import psutil

# 检测 Google Colab 环境
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("=" * 80)
print("初始化 Notebook 环境")
print("=" * 80)

# 设置工作目录
if IN_COLAB:
    WORK_DIR = Path("/tmp/ceg_wm_workspace")
    print(f"✓ 检测到 Google Colab 环境")
else:
    WORK_DIR = Path.cwd()
    print(f"✓ 本地 Jupyter 环境")

WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"  工作目录：{WORK_DIR}")

# 显示系统信息
print("\n系统信息：")
print(f"  操作系统：{sys.platform}")
print(f"  Python 版本：{sys.version.split()[0]}")
print(f"  CPU 核心数：{psutil.cpu_count()}")
print(f"  总内存：{psutil.virtual_memory().total / (1024**3):.1f} GB")
print(f"  可用内存：{psutil.virtual_memory().available / (1024**3):.1f} GB")

print("\n✓ 环境初始化完成")

In [ ]:

import zipfile
from google.colab import files
import os

# 1. 上传 zip 文件
print("请上传您的 CEG-WM.zip 文件。")
uploaded = files.upload()

for fn in uploaded.keys():
    print(f'已上传文件："{fn}"，大小 {len(uploaded[fn]) / (1024*1024):.2f} MB')
    zip_filename = fn

    # 2. 解压 zip 文件
    if zip_filename.endswith('.zip'):
        try:
            with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
                zip_ref.extractall(str(WORK_DIR))
            print(f"✓ 文件 '{zip_filename}' 已成功解压缩到 {WORK_DIR}")
        except zipfile.BadZipFile:
            print(f"✗ 错误：文件 '{zip_filename}' 不是有效的 ZIP 文件。")
        except Exception as e:
            print(f"✗ 解压缩时发生错误: {e}")
    else:
        print(f"⚠ 文件 '{zip_filename}' 不是 ZIP 文件。跳过解压缩。")


In [ ]:

# 验证和定位 CEG-WM 代码库
from pathlib import Path

def find_ceg_wm_root(base_dir):
    """
    功能：查找 CEG-WM 根目录（智能路径发现）。
    """
    base_dir = Path(base_dir)
    
    # 检查：特征目录结构
    characteristic_dirs = ["main/cli", "configs", "scripts", "requirements.txt"]
    
    # 首先检查直接目录
    if all((base_dir / d).exists() for d in characteristic_dirs):
        return base_dir
    
    # 查找包含正确结构的子目录
    for subdir in sorted(base_dir.iterdir()):
        if subdir.is_dir() and not subdir.name.startswith('.'):
            if all((subdir / d).exists() for d in characteristic_dirs):
                return subdir
    
    # 尝试找任何包含 main/cli 的目录（最后的备选方案）
    for possible_root in base_dir.rglob("main"):
        if possible_root.is_dir():
            ceg_root = possible_root.parent
            if (ceg_root / "configs").exists() and (ceg_root / "scripts").exists():
                return ceg_root
    
    raise FileNotFoundError(
        f"无法找到 CEG-WM 根目录\n"
        f"  搜索路径：{base_dir}\n"
        f"  期望目录结构：main/cli/, configs/, scripts/, requirements.txt\n"
        f"  请检查上传的代码包是否完整"
    )

# 定位 CEG-WM 根目录
try:
    CEG_WM_ROOT = find_ceg_wm_root(WORK_DIR)
    print(f"✓ 找到 CEG-WM 根目录：{CEG_WM_ROOT}")
    print(f"  绝对路径：{CEG_WM_ROOT.resolve()}")
except FileNotFoundError as e:
    print(f"✗ {e}")
    # 列出实际的目录结构以帮助调试
    print("\n实际目录结构：")
    for item in WORK_DIR.iterdir():
        if item.is_dir() and not item.name.startswith('.'):
            print(f"  {item.name}/")
            for subitem in item.iterdir():
                if subitem.is_dir():
                    print(f"    {subitem.name}/")


In [ ]:

# 添加 CEG-WM 到 Python 路径
if str(CEG_WM_ROOT) not in sys.path:
    sys.path.insert(0, str(CEG_WM_ROOT))

# 验证关键模块可导入
print("验证关键模块导入...")
try:
    from main.cli import run_embed
    print("  ✓ main.cli.run_embed 导入成功")
except ImportError as e:
    print(f"  ✗ 导入失败：{e}")
    print("    这表示 CEG-WM 依赖未完整安装，将在下一步修复")


## 第 2 步：安装依赖包

安装 CEG-WM 项目本身和所有依赖。这是关键步骤！

In [ ]:

import subprocess
import sys

print("=" * 80)
print("安装依赖包")
print("=" * 80)

# 步骤 1：升级 pip
print("\n[1/3] 升级 pip...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"], 
               capture_output=True)
print("  ✓ pip 已升级")

# 步骤 2：安装系统依赖
if IN_COLAB:
    print("\n[2/3] 安装系统依赖...")
    subprocess.run(["apt-get", "update", "-qq"], capture_output=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "git", "wget", "unzip"], 
                   capture_output=True)
    print("  ✓ 系统依赖已安装")

# 步骤 3：安装 CEG-WM 项目本身（关键！）
print("\n[3/3] 安装 CEG-WM 项目...")
try:
    # 方式 A：从 pyproject.toml（推荐）
    pyproject_file = CEG_WM_ROOT / "pyproject.toml"
    requirements_file = CEG_WM_ROOT / "requirements.txt"
    
    if pyproject_file.exists():
        print(f"  从 pyproject.toml 安装：{pyproject_file}")
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-e", str(CEG_WM_ROOT)],
            capture_output=True,
            text=True,
            timeout=300
        )
        if result.returncode == 0:
            print("  ✓ 项目安装成功（开发模式）")
        else:
            print(f"  ⚠ 项目安装失败，尝试备选方案")
            print(f"    错误：{result.stderr[-200:]}")
    
    elif requirements_file.exists():
        print(f"  从 requirements.txt 安装：{requirements_file}")
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-r", str(requirements_file)],
            capture_output=True,
            text=True,
            timeout=300
        )
        if result.returncode == 0:
            print("  ✓ 依赖安装成功")
        else:
            print(f"  ⚠ 部分依赖安装失败，继续使用...")
    else:
        print("  ⚠ 未找到 pyproject.toml 或 requirements.txt")
        
except subprocess.TimeoutExpired:
    print("  ⚠ 安装超时（300 秒），但继续执行")
except Exception as e:
    print(f"  ⚠ 安装出错：{e}，但继续执行")

print("\n✓ 依赖安装步骤完成")


## 第 3 步：下载模型权重

真实下载 Stable Diffusion 3 模型。这是关键步骤！

In [ ]:
import time
import random
import os
import gc
import torch
from pathlib import Path
from diffusers import StableDiffusion3Pipeline
from huggingface_hub import scan_cache_dir

print("=" * 80)
print("下载模型权重")
print("=" * 80)

# 配置模型缓存目录
MODEL_CACHE_DIR = WORK_DIR / "huggingface_cache"
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(MODEL_CACHE_DIR)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(MODEL_CACHE_DIR)

print(f"\n模型缓存目录：{MODEL_CACHE_DIR}")
print(f"  可用磁盘空间：{os.statvfs(str(MODEL_CACHE_DIR)).f_bavail * os.statvfs(str(MODEL_CACHE_DIR)).f_frsize / (1024**3):.1f} GB")

print("\n检查 Hugging Face 认证...")
try:
    from huggingface_hub import HfApi
    api = HfApi()
    try:
        user_info = api.whoami()
        print(f"  ✓ 已认证：{user_info.get('name', 'Unknown')}")
    except:
        print("  ℹ 未认证（使用匿名访问）")
except ImportError:
    print("  ⚠ huggingface_hub 未安装")

# 下载 Stable Diffusion 3.5 模型
print("\n" + "=" * 80)
print("下载 Stable Diffusion 3.5 模型")
print("=" * 80)

MODEL_ID = "stabilityai/stable-diffusion-3.5-medium"

# 检查模型是否已缓存
def check_model_cached(model_id):
    """
    功能：检查模型是否已在缓存中。
    
    Check if model is already cached locally.
    
    Args:
        model_id: Model identifier.
        
    Returns:
        Boolean indicating if model is cached.
    """
    try:
        cache_info = scan_cache_dir()
        for repo in cache_info.repos:
            if model_id in repo.repo_id:
                return True
        return False
    except Exception:
        return False

if check_model_cached(MODEL_ID):
    print(f"✓ 模型已缓存：{MODEL_ID}")
    print("  跳过下载")
else:
    print(f"\n⏳ 模型未缓存，开始下载：{MODEL_ID}")
    print("   这可能需要 5-15 分钟（取决于网络）")
    
    try:
        # 下载模型到本地缓存（仅缓存，不加载到内存）
        print("  下载中...")
        
        # 注意：StableDiffusion3Pipeline 不支持 device_map="cpu"
        # 仅执行模型下载和缓存，使用平衡策略避免 OOM
        pipeline = StableDiffusion3Pipeline.from_pretrained(
            MODEL_ID,
            cache_dir=str(MODEL_CACHE_DIR),
            torch_dtype=torch.float16,  # 使用 float16 节省显存
            device_map="balanced"  # 使用平衡策略（自动分配到可用设备）
        )
        print("  ✓ 模型下载完成")
        
        # 验证模型加载
        print("  验证模型结构...")
        print(f"    - Text encoder: {type(pipeline.text_encoder).__name__}")
        print(f"    - Transformer: {type(pipeline.transformer).__name__}")
        print(f"    - VAE: {type(pipeline.vae).__name__}")
        print(f"    - Scheduler: {type(pipeline.scheduler).__name__}")
        
    except Exception as e:
        error_msg = str(e)
        print(f"  ✗ 模型下载失败：{e}")
        
        # 检查是否是访问权限问题
        if "404" in error_msg or "Entry Not Found" in error_msg:
            print("\n  ❌ 错误：无法访问模型（404）")
            print("  原因：SD3.5 是受限访问模型，需要显式授权")
            print("\n  解决方案：")
            print("    1. 访问 https://huggingface.co/stabilityai/stable-diffusion-3.5-medium")
            print("    2. 同意使用条款（点击 'Access repository'）")
            print("    3. 获取 HF_TOKEN：https://huggingface.co/settings/tokens")
            print("    4. 在下方 cell 中设置 HF_TOKEN：")
            print("       os.environ['HF_TOKEN'] = 'your_token_here'")
        elif "401" in error_msg or "Unauthorized" in error_msg:
            print("\n  ❌ 错误：认证失败（401）")
            print("  原因：HF_TOKEN 无效或已过期")
            print("\n  解决方案：")
            print("    1. 检查 HF_TOKEN 是否正确")
            print("    2. 获取新 token：https://huggingface.co/settings/tokens")
            print("    3. 使用新 token 重新运行此 cell")
        else:
            print("\n  解决方案：")
            print("    1. 检查网络连接")
            print("    2. 检查 HF_TOKEN 有效性")
            print("    3. 检查 GPU 可用性")
            print("    4. 尝试使用代理或 HF 镜像")
        
        print("\n  ⚠️ 继续执行（后续步骤可能失败）...")

# 清理内存
print("\n清理内存...")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("  ✓ GPU 缓存已清理")

print("✓ 模型准备完成")

## 第 4 步：配置选择和数据准备

选择工作流配置并准备输入数据。

In [ ]:
import json

print("=" * 80)
print("工作流配置和数据准备")
print("=" * 80)

# 选择配置文件
CONFIG_CHOICE = "paper_proof"  # 可选：'default' 或 'paper_proof'
print(f"\n选定配置：{CONFIG_CHOICE}")

CONFIG_FILE = CEG_WM_ROOT / "configs" / f"{CONFIG_CHOICE}.yaml"
if not CONFIG_FILE.exists():
    print(f"⚠ 配置文件不存在：{CONFIG_FILE}")
    # 尝试找到可用的配置
    available_configs = list((CEG_WM_ROOT / "configs").glob("*.yaml"))
    if available_configs:
        CONFIG_FILE = available_configs[0]
        CONFIG_CHOICE = CONFIG_FILE.stem
        print(f"  使用备选配置：{CONFIG_CHOICE}")
    else:
        print("✗ 没有找到任何配置文件")

print(f"  配置文件路径：{CONFIG_FILE}")
print(f"  配置文件大小：{CONFIG_FILE.stat().st_size / 1024:.1f} KB")

# 创建数据目录
data_dir = CEG_WM_ROOT / "data"
data_dir.mkdir(parents=True, exist_ok=True)

# 准备输入数据（修复版）
print("\n准备输入数据...")

## 第 5-8 步：分阶段执行完整工作流

已拆分为 4 个独立 code cell，按顺序执行：

- 第 5 步：Embed
- 第 6 步：Detect
- 第 7 步：Calibrate
- 第 8 步：Evaluate

每个阶段均单独输出日志到 `RUN_ROOT/logs/*_stage.log`，便于快速定位失败点。

In [ ]:
import os
import sys
import subprocess
import shutil
from datetime import datetime

print("=" * 80)
print("第 5 步：执行 Embed 阶段")
print("=" * 80)

if 'CEG_WM_ROOT' not in globals() or 'CONFIG_FILE' not in globals() or 'CONFIG_CHOICE' not in globals():
    raise RuntimeError("请先执行前面的环境与配置 cell（第 1-4 步）")

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
if 'OMP_NUM_THREADS' not in os.environ:
    os.environ['OMP_NUM_THREADS'] = str(min(4, psutil.cpu_count() or 4))

RUN_ROOT = CEG_WM_ROOT / "outputs" / f"colab_run_{CONFIG_CHOICE}"
RUN_ROOT.mkdir(parents=True, exist_ok=True)

old_records = RUN_ROOT / "records"
old_artifacts = RUN_ROOT / "artifacts"
old_logs = RUN_ROOT / "logs"
if old_records.exists() or old_artifacts.exists() or old_logs.exists():
    print("\n清理旧的运行数据...")
    if old_records.exists():
        shutil.rmtree(old_records)
    if old_artifacts.exists():
        shutil.rmtree(old_artifacts)
    if old_logs.exists():
        shutil.rmtree(old_logs)
    print("  ✓ 旧数据已清理")

logs_dir = RUN_ROOT / "logs"
logs_dir.mkdir(parents=True, exist_ok=True)

PIPELINE_OVERRIDES = []
if 'FORCE_PIPELINE_OVERRIDES' in globals() and isinstance(FORCE_PIPELINE_OVERRIDES, list):
    PIPELINE_OVERRIDES.extend(FORCE_PIPELINE_OVERRIDES)

ACTIVE_CONFIG_FILE = CONFIG_FILE

print("\nEmbed 执行参数：")
print(f"  输出目录：{RUN_ROOT}")
print(f"  原始配置文件：{CONFIG_FILE.name}")
print(f"  初始 override：{PIPELINE_OVERRIDES}")

def clean_stage_dirs_for_retry():
    """Clean stage output dirs before retry to satisfy path policy."""
    records_dir = RUN_ROOT / "records"
    artifacts_dir = RUN_ROOT / "artifacts"
    logs_path = RUN_ROOT / "logs"
    if records_dir.exists():
        shutil.rmtree(records_dir)
    if artifacts_dir.exists():
        shutil.rmtree(artifacts_dir)
    if logs_path.exists():
        shutil.rmtree(logs_path)
    logs_path.mkdir(parents=True, exist_ok=True)
    return logs_path

def get_embed_fallback_config(config_choice):
    """Resolve embed fallback config file without generating YAML in notebook."""
    fallback_name = f"colab_embed_fallback_{config_choice}.yaml"
    fallback_path = CEG_WM_ROOT / "configs" / fallback_name
    if not fallback_path.exists():
        raise RuntimeError(f"未找到回退配置文件：{fallback_path}")
    return fallback_path

def run_stage(stage_name, stage_overrides=None, timeout_seconds=3600):
    """Run a single stage command and persist logs for debugging."""
    if stage_overrides is None:
        stage_overrides = []

    command = [
        sys.executable,
        "-m",
        f"main.cli.run_{stage_name}",
        "--out",
        str(RUN_ROOT),
        "--config",
        str(ACTIVE_CONFIG_FILE),
    ]
    for item in stage_overrides:
        command.extend(["--override", item])

    print("\n" + "-" * 80)
    print(f"[{stage_name}] 命令：")
    print("  " + " ".join(command))
    print(f"[{stage_name}] 开始时间：{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

    result = subprocess.run(
        command,
        cwd=str(CEG_WM_ROOT),
        capture_output=True,
        text=True,
        timeout=timeout_seconds,
    )

    log_file = logs_dir / f"{stage_name}_stage.log"
    with open(log_file, "w", encoding="utf-8") as f:
        f.write("COMMAND:\n")
        f.write(" ".join(command))
        f.write("\n\nSTDOUT:\n")
        f.write(result.stdout or "")
        f.write("\n\nSTDERR:\n")
        f.write(result.stderr or "")

    print(f"[{stage_name}] 结束时间：{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"[{stage_name}] 返回码：{result.returncode}")
    print(f"[{stage_name}] 日志：{log_file}")

    if result.stdout:
        print(f"[{stage_name}] STDOUT（最后 50 行）：")
        for line in result.stdout.splitlines()[-50:]:
            print(line)
    if result.stderr:
        print(f"[{stage_name}] STDERR（最后 30 行）：")
        for line in result.stderr.splitlines()[-30:]:
            print(line)

    return result

def print_embed_debug_paths():
    """Print key debug paths when embed stage fails."""
    records_dir = RUN_ROOT / "records"
    artifacts_dir = RUN_ROOT / "artifacts"
    embed_log_file = logs_dir / "embed_stage.log"
    workflow_log_file = RUN_ROOT / "workflow_execution.log"

    print("\n关键排错路径：")
    print(f"  - RUN_ROOT: {RUN_ROOT}")
    print(f"  - records/: {records_dir}（存在={records_dir.exists()}）")
    print(f"  - artifacts/: {artifacts_dir}（存在={artifacts_dir.exists()}）")
    print(f"  - logs/: {logs_dir}（存在={logs_dir.exists()}）")
    print(f"  - embed_stage.log: {embed_log_file}（存在={embed_log_file.exists()}）")
    print(f"  - workflow_execution.log: {workflow_log_file}（存在={workflow_log_file.exists()}）")

    print("\n建议优先检查：")
    print("  1. embed_stage.log 中的最后 100 行")
    print("  2. records/embed_record.json 是否产出")
    print("  3. artifacts/run_closure.json 中的 status_details")

EMBED_STATUS = "unknown"
EMBED_USED_AUTO_FIX = False
embed_debug_paths_printed = False

try:
    embed_timeout_seconds = 7200 if CONFIG_CHOICE == "paper_proof" else 3600
    embed_result = run_stage("embed", PIPELINE_OVERRIDES, timeout_seconds=embed_timeout_seconds)

    if embed_result.returncode != 0 and "paper_faithfulness mode requires injection_evidence.status=ok" in (embed_result.stderr or ""):
        print("\n检测到 paper_faithfulness 注入证据校验失败，切换到回退配置...")
        ACTIVE_CONFIG_FILE = get_embed_fallback_config(CONFIG_CHOICE)
        CONFIG_FILE = ACTIVE_CONFIG_FILE
        EMBED_USED_AUTO_FIX = True
        print(f"  回退配置：{ACTIVE_CONFIG_FILE}")

        logs_dir = clean_stage_dirs_for_retry()
        print("  已清理 records/artifacts/logs，准备重试 Embed")

        embed_result = run_stage("embed", PIPELINE_OVERRIDES, timeout_seconds=embed_timeout_seconds)

    if embed_result.returncode == 0:
        EMBED_STATUS = "ok"
        print("\n✓ Embed 阶段执行成功")
        print(f"  当前生效配置：{CONFIG_FILE}")
        if EMBED_USED_AUTO_FIX:
            print("  注意：已切换为回退配置文件")
    else:
        EMBED_STATUS = "failed"
        print_embed_debug_paths()
        embed_debug_paths_printed = True
        raise RuntimeError(f"Embed 阶段失败，返回码={embed_result.returncode}。请查看 logs/embed_stage.log")

except subprocess.TimeoutExpired:
    EMBED_STATUS = "timeout"
    print("\n✗ Embed 阶段执行超时")
    if not embed_debug_paths_printed:
        print_embed_debug_paths()
        embed_debug_paths_printed = True
except Exception as e:
    EMBED_STATUS = "failed"
    print(f"\n✗ Embed 阶段执行出错：{type(e).__name__}: {e}")
    if not embed_debug_paths_printed:
        print_embed_debug_paths()
        embed_debug_paths_printed = True

print(f"\nEmbed 阶段状态：{EMBED_STATUS}")
print(f"当前统一 override：{PIPELINE_OVERRIDES}")

In [ ]:
import subprocess
from datetime import datetime

print("=" * 80)
print("第 6 步：执行 Detect 阶段")
print("=" * 80)

if 'RUN_ROOT' not in globals() or 'CONFIG_FILE' not in globals() or 'PIPELINE_OVERRIDES' not in globals():
    raise RuntimeError("请先执行第 5 步 Embed cell")
if globals().get('EMBED_STATUS') != 'ok':
    raise RuntimeError("Embed 阶段未成功，请先修复并重跑 Embed")

ACTIVE_CONFIG_FILE = CONFIG_FILE
print(f"Detect 阶段配置：{ACTIVE_CONFIG_FILE}")

detect_stage_overrides = list(PIPELINE_OVERRIDES)
detect_stage_overrides.extend([
    "run_root_reuse_allowed=true",
    "run_root_reuse_reason=\"colab_split_stage_detect\"",
])

DETECT_STATUS = "unknown"
try:
    detect_timeout_seconds = 3600
    detect_result = run_stage("detect", detect_stage_overrides, timeout_seconds=detect_timeout_seconds)
    if detect_result.returncode == 0:
        DETECT_STATUS = "ok"
        print("\n✓ Detect 阶段执行成功")
    else:
        DETECT_STATUS = "failed"
        raise RuntimeError(f"Detect 阶段失败，返回码={detect_result.returncode}。请查看 logs/detect_stage.log")
except subprocess.TimeoutExpired:
    DETECT_STATUS = "timeout"
    print("\n✗ Detect 阶段执行超时")
except Exception as e:
    DETECT_STATUS = "failed"
    print(f"\n✗ Detect 阶段执行出错：{type(e).__name__}: {e}")

print(f"\nDetect 阶段状态：{DETECT_STATUS}")
print(f"结束时间：{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
import subprocess
from datetime import datetime

print("=" * 80)
print("第 7 步：执行 Calibrate 阶段")
print("=" * 80)

if 'RUN_ROOT' not in globals() or 'CONFIG_FILE' not in globals() or 'PIPELINE_OVERRIDES' not in globals():
    raise RuntimeError("请先执行第 5 步 Embed cell")
if globals().get('EMBED_STATUS') != 'ok':
    raise RuntimeError("Embed 阶段未成功，请先修复并重跑 Embed")

ACTIVE_CONFIG_FILE = CONFIG_FILE
print(f"Calibrate 阶段配置：{ACTIVE_CONFIG_FILE}")

calibrate_stage_overrides = list(PIPELINE_OVERRIDES)
calibrate_stage_overrides.extend([
    "run_root_reuse_allowed=true",
    "run_root_reuse_reason=\"colab_split_stage_calibrate\"",
])

CALIBRATE_STATUS = "unknown"
try:
    calibrate_timeout_seconds = 3600
    calibrate_result = run_stage("calibrate", calibrate_stage_overrides, timeout_seconds=calibrate_timeout_seconds)
    if calibrate_result.returncode == 0:
        CALIBRATE_STATUS = "ok"
        print("\n✓ Calibrate 阶段执行成功")
    else:
        CALIBRATE_STATUS = "failed"
        raise RuntimeError(f"Calibrate 阶段失败，返回码={calibrate_result.returncode}。请查看 logs/calibrate_stage.log")
except subprocess.TimeoutExpired:
    CALIBRATE_STATUS = "timeout"
    print("\n✗ Calibrate 阶段执行超时")
except Exception as e:
    CALIBRATE_STATUS = "failed"
    print(f"\n✗ Calibrate 阶段执行出错：{type(e).__name__}: {e}")

print(f"\nCalibrate 阶段状态：{CALIBRATE_STATUS}")
print(f"结束时间：{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
import subprocess
from datetime import datetime

print("=" * 80)
print("第 8 步：执行 Evaluate 阶段")
print("=" * 80)

if 'RUN_ROOT' not in globals() or 'CONFIG_FILE' not in globals() or 'PIPELINE_OVERRIDES' not in globals():
    raise RuntimeError("请先执行第 5 步 Embed cell")
if globals().get('EMBED_STATUS') != 'ok':
    raise RuntimeError("Embed 阶段未成功，请先修复并重跑 Embed")

ACTIVE_CONFIG_FILE = CONFIG_FILE
print(f"Evaluate 阶段配置：{ACTIVE_CONFIG_FILE}")

evaluate_stage_overrides = list(PIPELINE_OVERRIDES)
evaluate_stage_overrides.extend([
    "run_root_reuse_allowed=true",
    "run_root_reuse_reason=\"colab_split_stage_evaluate\"",
])

EVALUATE_STATUS = "unknown"
try:
    evaluate_timeout_seconds = 7200 if CONFIG_CHOICE == "paper_proof" else 3600
    evaluate_result = run_stage("evaluate", evaluate_stage_overrides, timeout_seconds=evaluate_timeout_seconds)
    if evaluate_result.returncode == 0:
        EVALUATE_STATUS = "ok"
        print("\n✓ Evaluate 阶段执行成功")
    else:
        EVALUATE_STATUS = "failed"
        raise RuntimeError(f"Evaluate 阶段失败，返回码={evaluate_result.returncode}。请查看 logs/evaluate_stage.log")
except subprocess.TimeoutExpired:
    EVALUATE_STATUS = "timeout"
    print("\n✗ Evaluate 阶段执行超时")
except Exception as e:
    EVALUATE_STATUS = "failed"
    print(f"\n✗ Evaluate 阶段执行出错：{type(e).__name__}: {e}")

print(f"\nEvaluate 阶段状态：{EVALUATE_STATUS}")
print(f"结束时间：{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 第 9 步：验证工作流输出

检查工作流是否成功生成了所有必要的输出文件。

In [ ]:
import json

print("=" * 80)
print("验证工作流输出")
print("=" * 80)

# 验证记录文件（records/ 目录）
print("\n[1/3] 检查记录文件...")
records_dir = RUN_ROOT / "records"
expected_records = [
    "embed_record.json",    # Embed 阶段记录
    "detect_record.json",   # Detect 阶段记录
    "calibrate_record.json", # Calibrate 阶段记录
    "evaluate_record.json",  # Evaluate 阶段记录（可能不存在）
]

records_found = {}

## 第 10 步：结果打包和下载

将完整的 run_root 目录压缩并下载到本地计算机。

In [ ]:

import shutil
from pathlib import Path

print("=" * 80)
print("打包和下载结果")
print("=" * 80)

# 创建压缩包
ARCHIVE_NAME = f"ceg_wm_run_root_{CONFIG_CHOICE}"
ARCHIVE_PATH = WORK_DIR / f"{ARCHIVE_NAME}.zip"

print(f"\n压缩目录结构...")
print(f"  源目录：{RUN_ROOT}")
print(f"  目标文件：{ARCHIVE_PATH.name}")

try:
    # 使用 shutil.make_archive 进行压缩
    archive_without_ext = str(WORK_DIR / ARCHIVE_NAME)
    base_name = shutil.make_archive(
        archive_without_ext,
        'zip',
        RUN_ROOT.parent,
        RUN_ROOT.name
    )
    
    if ARCHIVE_PATH.exists():
        size_mb = ARCHIVE_PATH.stat().st_size / (1024 * 1024)
        print(f"✓ 压缩成功")
        print(f"  文件大小：{size_mb:.2f} MB")
        print(f"  完整路径：{ARCHIVE_PATH}")
    else:
        print("✗ 压缩失败（文件不存在）")
        ARCHIVE_PATH = None
        
except Exception as e:
    print(f"✗ 压缩过程出错：{e}")
    ARCHIVE_PATH = None

# 下载压缩包
if ARCHIVE_PATH and ARCHIVE_PATH.exists():
    print(f"\n下载压缩包...")
    
    if IN_COLAB:
        from google.colab import files
        try:
            files.download(str(ARCHIVE_PATH))
            print(f"✓ 文件已下载：{ARCHIVE_PATH.name}")
            print(f"  查看浏览器的下载文件夹")
        except Exception as e:
            print(f"✗ 下载失败：{e}")
    else:
        print(f"本地环境提示：")
        print(f"  压缩包位置：{ARCHIVE_PATH}")
        print(f"  可以直接从文件系统访问或下载")
else:
    print("✗ 无法下载（压缩包创建失败）")

print("\n✓ 打包和下载步骤完成")


## 最后：执行总结报告

生成详细的执行总结。

In [ ]:

import json
from datetime import datetime

print("\n" + "=" * 80)
print("执行总结")
print("=" * 80)

summary = {
    "timestamp": datetime.now().isoformat(),
    "environment": "Google Colab" if IN_COLAB else "Local Jupyter",
    "system_info": {
        "total_memory_gb": round(psutil.virtual_memory().total / (1024**3), 1),
        "available_memory_gb": round(psutil.virtual_memory().available / (1024**3), 1),
        "cpu_count": psutil.cpu_count(),
    },
    "config_used": CONFIG_CHOICE,
    "ceg_wm_root": str(CEG_WM_ROOT),
    "run_root": str(RUN_ROOT),
    "records": {
        "embed": records_found.get("embed_record.json", False),
        "detect": records_found.get("detect_record.json", False),
        "calibrate": records_found.get("calibrate_record.json", False),
        "evaluate": records_found.get("evaluate_record.json", False),
    },
    "artifacts": {
        "evaluation_report": artifacts_found.get("evaluation_report.json", False),
        "run_closure": artifacts_found.get("run_closure.json", False),
    },
    "archive": {
        "created": ARCHIVE_PATH is not None and ARCHIVE_PATH.exists(),
        "path": str(ARCHIVE_PATH) if ARCHIVE_PATH else None,
        "size_mb": round(ARCHIVE_PATH.stat().st_size / (1024 * 1024), 2) if (ARCHIVE_PATH and ARCHIVE_PATH.exists()) else None,
    },
}

# 显示总结
print("\n元数据：")
print(f"  执行时间：{summary['timestamp']}")
print(f"  运行环境：{summary['environment']}")
print(f"  配置：{summary['config_used']}")

print(f"\n系统资源：")
for key, value in summary['system_info'].items():
    print(f"  {key}: {value}")

print(f"\n生成的记录文件：")
for stage, exists in summary['records'].items():
    status = "✓" if exists else "✗"
    print(f"  {status} {stage}_record.json")

print(f"\n生成的工件文件：")
for artifact, exists in summary['artifacts'].items():
    status = "✓" if exists else "✗"
    print(f"  {status} {artifact}.json")

print(f"\n压缩包信息：")
if summary['archive']['created']:
    print(f"  ✓ 已创建：{summary['archive']['path']}")
    print(f"  大小：{summary['archive']['size_mb']} MB")
else:
    print(f"  ✗ 压缩包创建失败")

# 保存总结到文件
summary_file = RUN_ROOT / "execution_summary.json"
with open(summary_file, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f"\n执行总结已保存到：{summary_file}")

# 显示 run_root 目录树
print(f"\nrun_root 目录结构（树形表示）：")
def show_tree(directory, prefix="", max_items=5, max_depth=3, current_depth=0):
    """显示目录树"""
    if current_depth >= max_depth:
        return
    
    try:
        items = sorted(directory.iterdir())
    except PermissionError:
        return
    
    dirs = [item for item in items if item.is_dir() and not item.name.startswith('.')]
    files = [item for item in items if item.is_file()]
    
    # 显示目录
    for i, item in enumerate(dirs[:max_items]):
        is_last = (i == len(dirs) - 1) and len(files) == 0
        print(f"{prefix}{'└── ' if is_last else '├── '}{item.name}/")
        extension = "    " if is_last else "│   "
        show_tree(item, prefix + extension, max_items=3, max_depth=max_depth, current_depth=current_depth + 1)
    
    # 显示文件
    for i, item in enumerate(files[:3]):
        is_last = i == min(2, len(files) - 1)
        size_kb = item.stat().st_size / 1024
        print(f"{prefix}{'└── ' if is_last else '├── '}{item.name} ({size_kb:.1f} KB)")

print(f"{RUN_ROOT.name}/")
show_tree(RUN_ROOT)

print("\n" + "=" * 80)
print("✓ Colab 工作流执行完成！")
print("=" * 80)
print("\n下一步：")
if ARCHIVE_PATH and ARCHIVE_PATH.exists():
    print(f"  1. 已下载压缩包：{ARCHIVE_PATH.name}")
    print(f"  2. 在本地解压缩")
    print(f"  3. 查看 run_root/ 中的结果文件")
else:
    print(f"  1. 检查 run_root 目录：{RUN_ROOT}")
    print(f"  2. 查看工作流日志：{RUN_ROOT / 'workflow_execution.log'}")


## 配置选项参考

CEG-WM 支持多种配置文件，放在 `configs/` 目录下：

| 配置文件 | 用途 | 说明 |
|---------|------|------|
| `default.yaml` | 基础配置 | 最小化设置，快速测试 |
| `paper_proof.yaml` | 论文验证 | 启用完整功能（几何链、NP 融合、latent per-step） |
| `model_sd3.yaml` | SD3 模型配置 | Stable Diffusion 3 特定参数 |
| `paper_faithfulness_spec.yaml` | 论文保真度 | 保真度相关参数 |

### 切换配置

修改 notebook 中的配置选择：
```python
CONFIG_CHOICE = "paper_proof"  # 或 "default"
```

## 输出结构参考

执行完成后，`run_root` 目录包含以下结构：

```
run_root/
├── artifacts/
│   ├── embed_output.json           # Embed 阶段输出
│   ├── detect_output.json          # Detect 阶段输出
│   ├── calibrate_output.json       # Calibrate 阶段输出
│   ├── evaluation_report.json      # Evaluate 阶段完整报告
│   ├── run_closure.json            # 运行闭包（元数据）
│   ├── cfg_audit/                  # 配置审计信息
│   ├── env_audits/                 # 环境审计信息
│   └── repro_bundle/               # 可复现性证据包
├── intermediate/
│   ├── embed_cache/                # Embedding 中间缓存
│   ├── detect_cache/               # Detection 中间缓存
│   └── ...
└── logs/
    └── workflow.log                # 完整执行日志
```

### 关键输出文件说明

- **evaluation_report.json**：包含完整的评估指标和模型性能统计
- **run_closure.json**：记录运行完整性和完整性检查结果
- **repro_bundle/**：包含可复现性所需的所有信息（config_digest、plan_digest 等）